# 03 · Senescence scoring

SenePy scoring · reference-anchored threshold · depth diagnostic

**In** — `{dataset}_pearson.h5ad` (module 02)
**Out** — `{dataset}_scored.h5ad`
- `.X` = log-normalized (activated for downstream)
- `obs['senescence_score']`, `obs['is_senescent']`, `obs['Study_Group']`

`is_senescent` is the most-consumed variable in the pipeline. Two decisions here
determine everything downstream: what the score is computed on, and where the
threshold is anchored.

**Environment:** this module needs the `senepy` conda env. Modules 01, 02 and 04
run under `omicverse`.

## Config

In [ ]:
# ===========================================================================
# CONFIG
# ===========================================================================
import os
from pathlib import Path

DATASET = 'psychad_aging'    # psychad_aging | psychad_ad | psychencode | mathys

DATA_ROOT = Path(os.environ.get('SENESCENCE_DATA', 'data'))

# -- scoring ----------------------------------------------------------------
SENEPY_SPECIES = 'Human'
SENEPY_TISSUE  = 'hippocampus'   # closest brain calibration; SenePy has no cortical hub
SD_THRESHOLD   = 2.0             # threshold = mean + SD_THRESHOLD*sd of the reference group

# -- columns ----------------------------------------------------------------
CELL_TYPE_COLUMN   = 'subclass'
AGE_COLUMN         = 'Age'
SEX_COLUMN         = 'Sex'
DONOR_COLUMN       = 'Sample'
STUDY_GROUP_COLUMN = 'Study_Group'

# -- per-dataset: how Study_Group is built, where the threshold anchors ------
DATASET_CONFIG = {
    'psychad_aging': {'type': 'aging',   'reference_group': 'Age_20_29',
                      'age_bins': [(20,29),(30,39),(40,49),(50,59),(60,69),(70,79),(80,100)],
                      'diagnosis_column': None},
    'psychad_ad':    {'type': 'disease', 'reference_group': 'Old_Healthy_Control',
                      'age_bins': None, 'diagnosis_column': 'Disease_Group'},
    'psychencode':   {'type': 'aging',   'reference_group': 'Age_30_39',
                      'age_bins': [(30,39),(40,49),(50,59),(60,69),(70,79),(80,100)],
                      'diagnosis_column': None},
    'mathys':        {'type': 'disease', 'reference_group': 'NCI',
                      'age_bins': None, 'diagnosis_column': 'Disease_Group'},
}
cfg             = DATASET_CONFIG[DATASET]
MODE            = cfg['type']              # aging | disease
REFERENCE_GROUP = cfg['reference_group']

# -- output size ------------------------------------------------------------
# Residuals exist only to be scored. Once scoring is done they are dead weight in
# a file every downstream module loads. Set True only to audit the scoring.
KEEP_RESIDUAL_LAYER = False

INPUT_FILE  = DATA_ROOT / 'processed' / f'{DATASET}_pearson.h5ad'
OUTPUT_FILE = DATA_ROOT / 'processed' / f'{DATASET}_scored.h5ad'
FIGURES_DIR = DATA_ROOT / 'figures' / '03_senescence_scoring' / DATASET
RESULTS_DIR = DATA_ROOT / 'results' / '03_senescence_scoring' / DATASET
for d in (OUTPUT_FILE.parent, FIGURES_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 0

print(f"  dataset   : {DATASET}   mode: {MODE}")
print(f"  threshold : mean + {SD_THRESHOLD}*sd of '{REFERENCE_GROUP}', per cell type")
print(f"  hub       : {SENEPY_SPECIES} / {SENEPY_TISSUE}")
print(f"  in        : {INPUT_FILE}")
print(f"  out       : {OUTPUT_FILE}")

## Setup

In [ ]:
import platform
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.linear_model import LinearRegression
import warnings; warnings.filterwarnings('ignore')

import senepy as sp_py                    # senepy env only
np.random.seed(SEED)
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 10,
    'axes.labelsize': 10, 'axes.titlesize': 11, 'legend.fontsize': 9,
    'font.family': 'sans-serif', 'axes.linewidth': 1.0,
    'axes.grid': False, 'pdf.fonttype': 42,
})

(RESULTS_DIR / 'run_info.txt').write_text(
    f"module    : 03_senescence_scoring\ndataset   : {DATASET}\nmode      : {MODE}\n"
    f"hub       : {SENEPY_SPECIES}/{SENEPY_TISSUE}\nsd        : {SD_THRESHOLD}\n"
    f"reference : {REFERENCE_GROUP}\npython    : {platform.python_version()}\n"
    f"scanpy    : {sc.__version__}\n")

print(f"scanpy {sc.__version__} - senepy loaded")

## Load and build Study_Group

**Why.** `Study_Group` is the column that lets one threshold rule serve both arms:
decade bins for aging cohorts, diagnosis for disease cohorts. Everything after
this point reads `Study_Group` and does not need to know which arm it is in.

The reference group must exist in it - that is what the threshold anchors on, so
a missing reference silently changes the definition of "senescent".

In [ ]:
adata = sc.read_h5ad(INPUT_FILE)
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"  layers: {list(adata.layers.keys())}")

missing = [c for c in (CELL_TYPE_COLUMN, AGE_COLUMN, SEX_COLUMN, DONOR_COLUMN)
           if c not in adata.obs.columns]
if missing:
    raise KeyError(f"missing required columns {missing}; have {list(adata.obs.columns)}")

_x = adata.X
if float(_x.data.min() if sp.issparse(_x) else _x.min()) >= 0:
    raise ValueError(".X has no negative values - expected Pearson residuals from module 02")
print("  .X confirmed as residuals (has negatives)")

In [ ]:
if MODE == 'aging':
    bins = cfg['age_bins']
    def _bin(age):
        for lo, hi in bins:
            if lo <= age <= hi:
                return f"Age_{lo}_{hi}"
        return "Age_Unknown"
    adata.obs[STUDY_GROUP_COLUMN] = adata.obs[AGE_COLUMN].apply(_bin)
    n_unk = (adata.obs[STUDY_GROUP_COLUMN] == 'Age_Unknown').sum()
    if n_unk:
        print(f"  WARNING {n_unk:,} cells fell outside all age bins")
else:
    dcol = cfg['diagnosis_column']
    if dcol not in adata.obs.columns:
        raise KeyError(f"diagnosis column '{dcol}' not found; have {list(adata.obs.columns)}")
    adata.obs[STUDY_GROUP_COLUMN] = adata.obs[dcol].astype(str)

print(adata.obs[STUDY_GROUP_COLUMN].value_counts().sort_index().to_string())

if REFERENCE_GROUP not in set(adata.obs[STUDY_GROUP_COLUMN]):
    raise ValueError(
        f"reference group '{REFERENCE_GROUP}' not present in {STUDY_GROUP_COLUMN}.\n"
        f"  available: {sorted(set(adata.obs[STUDY_GROUP_COLUMN]))}\n"
        f"  the threshold anchors on this group - it cannot be missing.")
print(f"\n  reference '{REFERENCE_GROUP}': "
      f"{(adata.obs[STUDY_GROUP_COLUMN] == REFERENCE_GROUP).sum():,} cells")

## SenePy scoring

**Why SenePy as the pre-specified primary panel.** It is built for scRNA-seq with
cell-type and sex-specific calibration. The alternatives - CellAge, Fridman,
Hernandez-Segura, SenMayo - were derived from bulk fibroblast data. That is the
reason for the choice, made before seeing results; the alternatives run in module
04 as robustness checks, not as competitors.

`identifiers=[cell_type, sex]` is what makes the scoring cell-type and
sex-specific - SenePy handles it internally, so it does not appear in the
threshold code below.

**Hippocampus hubs** are used because SenePy has no cortical hub. A documented
approximation applied to DLPFC tissue; state it in Methods. The hubs are merged
into one panel before scoring.

Scores are computed on `.X` - the Pearson residuals from module 02.

In [ ]:
hubs = sp_py.load_hubs(species=SENEPY_SPECIES)
tissue_hubs = hubs.metadata[hubs.metadata.tissue == SENEPY_TISSUE]
print(f"  {len(hubs.metadata)} hub entries - {len(tissue_hubs)} for '{SENEPY_TISSUE}'")
if len(tissue_hubs) == 0:
    raise ValueError(f"no hubs for tissue '{SENEPY_TISSUE}'; "
                     f"available: {sorted(hubs.metadata.tissue.unique())}")

hubs.merge_hubs(tissue_hubs, new_name='brain')
translator = sp_py.translator(hub=hubs.hubs, data=adata)

adata.obs['senescence_score'] = sp_py.score_all_cells(
    adata, hubs.hubs['brain'],
    identifiers=[CELL_TYPE_COLUMN, SEX_COLUMN],     # cell-type + sex specific
    translator=translator)

s = adata.obs['senescence_score']
print(f"\n  score: mean {s.mean():+.4f}  sd {s.std():.4f}  "
      f"range [{s.min():+.3f}, {s.max():+.3f}]")

## Threshold

**Why anchored on the reference group.** Threshold = mean + 2*sd of the reference
group *within each cell type*. Computing it on pooled data would let the threshold
drift with the very effect being measured - if disease raises scores, a pooled
threshold rises too and partially erases the difference.

**Do not tune `SD_THRESHOLD` to hit a target SnC%.** The realized fraction is a
property of the score distribution, not a construction. A mean+2sd cutoff yields
~2.3% only if scores are Gaussian, and single-cell senescence scores are
right-skewed. Adjusting SD until the fraction "looks right" silently converts an
anchored SD threshold into a quantile threshold. Sweep it systematically in
module 04 instead.

The source falls back to a pooled threshold when a cell type has no reference
cells. That is a silent change of definition for that cell type, so it now
reports explicitly and fails the gate.

In [ ]:
ref = adata.obs[STUDY_GROUP_COLUMN] == REFERENCE_GROUP
rows, fallback = [], []

for ct in adata.obs[CELL_TYPE_COLUMN].unique():
    in_ct  = adata.obs[CELL_TYPE_COLUMN] == ct
    ref_ct = adata.obs.loc[ref & in_ct, 'senescence_score']
    if len(ref_ct) > 0:
        thr, anchor, n_ref = ref_ct.mean() + SD_THRESHOLD*ref_ct.std(), 'reference', len(ref_ct)
    else:
        pooled = adata.obs.loc[in_ct, 'senescence_score']
        thr, anchor, n_ref = pooled.mean() + SD_THRESHOLD*pooled.std(), 'POOLED-FALLBACK', 0
        fallback.append(ct)
    rows.append({'cell_type': ct, 'threshold': thr, 'anchor': anchor,
                 'n_reference_cells': n_ref, 'n_cells': int(in_ct.sum())})

thresholds = pd.DataFrame(rows).set_index('cell_type')

adata.obs['is_senescent'] = False
for ct, r in thresholds.iterrows():
    m = (adata.obs[CELL_TYPE_COLUMN] == ct) & (adata.obs['senescence_score'] >= r['threshold'])
    adata.obs.loc[m, 'is_senescent'] = True
adata.obs['senescence_label'] = np.where(adata.obs['is_senescent'], 'SnC', 'Non-SnC')

thresholds['n_snc'] = [int(((adata.obs[CELL_TYPE_COLUMN] == ct) &
                            adata.obs['is_senescent']).sum()) for ct in thresholds.index]
thresholds['pct_snc'] = 100 * thresholds['n_snc'] / thresholds['n_cells']
thresholds.to_csv(RESULTS_DIR / 'thresholds.csv')
print(thresholds.to_string())

if fallback:
    print(f"\n  WARNING POOLED FALLBACK used for {len(fallback)} cell type(s): {fallback}")
    print("    Those thresholds are NOT reference-anchored - their SnC calls are")
    print("    defined differently from every other cell type. Report or exclude.")

print(f"\n  overall SnC: {adata.obs['is_senescent'].sum():,} "
      f"({100*adata.obs['is_senescent'].mean():.2f}%)")

## Depth diagnostic

**Why reported and not corrected.** Senescent cells carry higher sequencing depth
than non-senescent ones. Two readings are possible: technical residue, or biology
- senescent cells are larger and transcriptionally hyperactive, and SASP is a
high-output secretory program, so elevated RNA content is a *predicted* property.

The pipeline takes the second reading. Depth is already regressed out of the
SenePy input in module 02; regressing it out of the score as well would remove
real signal. `senescence_score_adjusted` is computed here **as a diagnostic** and
is deliberately not consumed by any downstream module - module 04 carries the
depth-matched downsampling analysis that tests this properly, without regressing
anything twice.

In [ ]:
umi = adata.obs['total_counts']
snc = adata.obs['is_senescent']

med_s, med_n = umi[snc].median(), umi[~snc].median()
u = mannwhitneyu(umi[snc], umi[~snc]).pvalue
print(f"  median UMI   SnC {med_s:>9,.0f}   non-SnC {med_n:>9,.0f}   "
      f"fold {med_s/med_n:.2f}x   p={u:.2e}")

rows = []
for ct, g in adata.obs.groupby(CELL_TYPE_COLUMN):
    if g['is_senescent'].sum() < 10:
        continue
    a, b = g.loc[g.is_senescent, 'total_counts'], g.loc[~g.is_senescent, 'total_counts']
    r = spearmanr(g['senescence_score'], np.log10(g['total_counts'])).statistic
    rows.append({'cell_type': ct, 'median_snc': a.median(), 'median_non': b.median(),
                 'fold': a.median()/b.median(), 'rho_score_vs_log10umi': r})
depth = pd.DataFrame(rows).sort_values('fold', ascending=False)
depth.to_csv(RESULTS_DIR / 'depth_diagnostic.csv', index=False)
print("\n" + depth.to_string(index=False))

In [ ]:
# DIAGNOSTIC ONLY - no downstream module reads these columns.
adata.obs['senescence_score_adjusted'] = np.nan
for ct, g in adata.obs.groupby(CELL_TYPE_COLUMN):
    if len(g) < 50:
        adata.obs.loc[g.index, 'senescence_score_adjusted'] = g['senescence_score']
        continue
    X = np.log10(g['total_counts'].values).reshape(-1, 1)
    y = g['senescence_score'].values
    ok = np.isfinite(X.ravel()) & np.isfinite(y)
    fit = LinearRegression().fit(X[ok], y[ok])
    adata.obs.loc[g.index, 'senescence_score_adjusted'] = y - fit.predict(X) + y.mean()

r_raw = spearmanr(adata.obs['senescence_score'], np.log10(umi)).statistic
r_adj = spearmanr(adata.obs['senescence_score_adjusted'], np.log10(umi)).statistic
print(f"  score ~ log10(UMI)   raw {r_raw:+.3f}   adjusted {r_adj:+.3f}")
print("  adjusted column is diagnostic; is_senescent uses the RAW score")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].violinplot([np.log10(umi[~snc]), np.log10(umi[snc])], showmedians=True)
ax[0].set(xticks=[1, 2], xticklabels=['non-SnC', 'SnC'], ylabel='log10 UMI',
          title=f'depth by call  ({med_s/med_n:.2f}x)')
ax[1].barh(depth['cell_type'], depth['fold'], color='#E15759')
ax[1].axvline(1, color='k', lw=.8); ax[1].set(xlabel='UMI fold (SnC / non-SnC)')
ax[2].barh(thresholds.index, thresholds['pct_snc'], color='#4E79A7')
ax[2].set(xlabel='% SnC')
for a in ax:
    for s_ in ('top', 'right'): a.spines[s_].set_visible(False)
fig.suptitle(f'{DATASET} - senescence calls and depth', y=1.03)
fig.tight_layout(); fig.savefig(FIGURES_DIR / 'calls_and_depth.pdf', bbox_inches='tight')
plt.show()

## Save

**Why `.X` switches to log-normalized.** The residuals existed to be scored, and
scoring is done. Every downstream module reads log-normalized expression, so
activating it as `.X` here means no downstream module has to know residuals ever
existed.

**Why the residual layer is dropped.** Unlike module 02's output, this file is
permanent: every module from 04 onward loads it. Carrying a dense residual matrix
multiplies the size of the most-read file in the pipeline. It is re-derivable by
re-running module 02.

In [ ]:
if KEEP_RESIDUAL_LAYER:
    adata.layers['pearson_residuals'] = adata.X.copy()
    print("  kept residuals -> layers['pearson_residuals']")
else:
    print("  residuals dropped (re-derivable via module 02)")

adata.X = adata.layers['lognorm'].copy()
adata.layers.pop('lognorm', None)          # now duplicated in .X
print(f"  .X <- log-normalized   layers: {list(adata.layers.keys())}")

adata.write_h5ad(OUTPUT_FILE)
print(f"  saved {OUTPUT_FILE.name}  ({OUTPUT_FILE.stat().st_size/1e9:.2f} GB)")

## Gate

In [ ]:
def _gate(label, ok, detail=''):
    print(f"  [{'OK  ' if ok else 'FAIL'}] {label}{'  - ' + detail if detail else ''}")
    if not ok:
        raise AssertionError(f"GATE FAILED: {label}. {detail}")
    return ok

_gate("is_senescent present", 'is_senescent' in adata.obs)
_gate("senescence_score present", 'senescence_score' in adata.obs)
_gate("Study_Group present", STUDY_GROUP_COLUMN in adata.obs)
_gate("layers['counts'] retained", 'counts' in adata.layers)
_gate(".X is log-normalized (non-negative)",
      float(adata.X.data.min() if sp.issparse(adata.X) else adata.X.min()) >= 0)
_gate("all thresholds reference-anchored", len(fallback) == 0,
      f"pooled fallback: {fallback}" if fallback else "")

print("\n  RECORDED - carry these into Methods, do not tune to them")
print(f"    realized SnC     : {100*adata.obs['is_senescent'].mean():.2f}%")
print(f"    UMI fold SnC/non : {med_s/med_n:.2f}x")
print(f"    score ~ depth    : rho {r_raw:+.3f}")

for ct, r in thresholds.iterrows():
    if r['n_snc'] < 50:
        print(f"    [WARN] {ct}: {int(r['n_snc'])} senescent cells - "
              f"below the floor for stable modelling downstream")
print("\n  -> module 04 tests whether these calls survive their assumptions")